# 06 — TeleOCR Fine-tuned Adapter Benchmark — Google Colab Pro
Evaluates each PEFT adapter checkpoint on the same validation set, selects lowest validation CER, then optionally runs the frozen test.

## 0. Install

In [ ]:
%pip install -q -U "kagglehub>=1.0.2" "jiwer>=4.0.0" pandas pillow
%pip install -q -U "transformers>=4.57.1,<5.0" "accelerate>=1.2.0" "peft>=0.15.0"

## 1. Data

In [ ]:
import os, json, time, random, platform, unicodedata, gc, math, shutil, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
from jiwer import cer, wer

from google.colab import drive
drive.mount('/content/drive')

# Optional Colab secret. KaggleHub can also prompt/authenticate through its normal flow.
try:
    from google.colab import userdata
    token = userdata.get('KAGGLE_API_TOKEN')
    if token:
        os.environ['KAGGLE_API_TOKEN'] = token
except Exception:
    pass

import kagglehub

SEED = 42
RAW_HANDLE = 'ntklinhfitus/uit-hwdb'
MANIFEST_HANDLE = 'ntklinhfitus/uit-hwdb-manifest'
PROJECT_ROOT = Path('/content/drive/MyDrive/vlm_handwriting_ocr')
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
print('PROJECT_ROOT =', PROJECT_ROOT)

In [ ]:
# Try a partial raw download first. Fall back to the full Kaggle dataset if the
# installed KaggleHub/runtime does not accept directory-level download.
try:
    raw_download = Path(kagglehub.dataset_download(RAW_HANDLE, path='UIT_HWDB_line'))
except Exception as e:
    print('Partial raw download unavailable, falling back to full dataset:', repr(e))
    raw_download = Path(kagglehub.dataset_download(RAW_HANDLE))
manifest_download = Path(kagglehub.dataset_download(MANIFEST_HANDLE))

def locate_raw_line_root(base: Path) -> Path:
    pool=[base]+[p for p in base.rglob('*') if p.is_dir()]
    candidates=[p for p in pool if (p/'train_data').is_dir() and (p/'test_data').is_dir()]
    assert candidates, f'Cannot locate UIT-HWDB-line train_data/test_data under {base}'
    candidates.sort(key=lambda p: ('UIT_HWDB_line' not in str(p), len(str(p))))
    return candidates[0]

def locate_manifest_root(base: Path) -> Path:
    pool=[base]+[p for p in base.rglob('*') if p.is_dir()]
    for p in pool:
        if all((p/f).exists() for f in ['train.csv','val.csv','test.csv']):
            return p
    raise FileNotFoundError(f'Cannot locate train.csv/val.csv/test.csv under {base}')

RAW_ROOT=locate_raw_line_root(raw_download)
MANIFEST_ROOT=locate_manifest_root(manifest_download)
print('RAW_ROOT      =',RAW_ROOT)
print('MANIFEST_ROOT =',MANIFEST_ROOT)

In [ ]:
train_df=pd.read_csv(MANIFEST_ROOT/'train.csv')
val_df=pd.read_csv(MANIFEST_ROOT/'val.csv')
test_df=pd.read_csv(MANIFEST_ROOT/'test.csv')
EXPECTED={'train':6346,'validation':682,'test':201}
assert len(train_df)==EXPECTED['train'],len(train_df)
assert len(val_df)==EXPECTED['validation'],len(val_df)
assert len(test_df)==EXPECTED['test'],len(test_df)
required={'writer_id','filename','relative_path','text'}
for name,frame in [('train',train_df),('validation',val_df),('test',test_df)]:
    missing=required-set(frame.columns)
    assert not missing,f'{name} missing columns: {missing}'
train_writers=set(train_df.writer_id); val_writers=set(val_df.writer_id); test_writers=set(test_df.writer_id)
assert train_writers.isdisjoint(val_writers)
assert train_writers.isdisjoint(test_writers)
assert val_writers.isdisjoint(test_writers)

def resolve_image_path(row):
    return RAW_ROOT/str(row['relative_path'])
for name,frame in [('train',train_df),('validation',val_df),('test',test_df)]:
    missing=[str(resolve_image_path(r)) for _,r in frame.iterrows() if not resolve_image_path(r).exists()]
    assert not missing,f'{name}: missing image paths, e.g. {missing[:3]}'
print(f'Train      : {len(train_df)} samples | {len(train_writers)} writers')
print(f'Validation : {len(val_df)} samples | {len(val_writers)} writers')
print(f'Test       : {len(test_df)} samples | {len(test_writers)} writers')
print('✅ Frozen writer-disjoint split verified.')

In [ ]:
def normalize_for_eval(text):
    # Strict OCR evaluation: Unicode NFC only.
    return unicodedata.normalize('NFC',str(text))

def compute_metrics(gt_list,pred_list):
    if len(gt_list)!=len(pred_list) or len(gt_list)==0:
        raise ValueError('GT/prediction lists must have the same non-zero length.')
    gt=[normalize_for_eval(x) for x in gt_list]
    pred=[normalize_for_eval(x) for x in pred_list]
    exact=sum(g==p for g,p in zip(gt,pred))/len(gt)
    return {'CER':float(cer(gt,pred)),'WER':float(wer(gt,pred)),'Exact_Line_Accuracy':float(exact),'N':int(len(gt))}
assert compute_metrics(['Việt Nam'],['Việt Nam'])['CER']==0.0
assert compute_metrics(['Biển Đông.'],['Biển đông.'])['CER']>0.0
SMOKE_SIZE=20
smoke_df=val_df.sample(n=SMOKE_SIZE,random_state=SEED).sort_index().reset_index(drop=True)
print('✅ Strict evaluator + fixed 20-sample validation smoke set ready.')

## 2. Model/checkpoint setup

In [ ]:
import torch
from transformers import AutoProcessor
try:
    from transformers import AutoModelForMultimodalLM
    ModelClass=AutoModelForMultimodalLM
except ImportError:
    from transformers import AutoModel
    ModelClass=AutoModel
from peft import PeftModel
MODEL_ID='StarDoc-AI/TeleOCR'; SYSTEM_PROMPT='You are a helpful assistant.'; OCR_PROMPT='Please output the text content from the image.'; MAX_NEW_TOKENS=512
GPU_NAME=torch.cuda.get_device_name(0); DTYPE=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
CKPT_ROOT=PROJECT_ROOT/'checkpoints'/'teleocr_lora'; RESULT_DIR=PROJECT_ROOT/'results'/'teleocr'/'finetuned'; RESULT_DIR.mkdir(parents=True,exist_ok=True)
checkpoints=sorted([p for p in CKPT_ROOT.glob('checkpoint-*') if p.is_dir() and (p/'adapter_config.json').exists()],key=lambda p:int(p.name.split('-')[-1]))
if not checkpoints and (CKPT_ROOT/'adapter_config.json').exists(): checkpoints=[CKPT_ROOT]
assert checkpoints,f'No adapter checkpoints found in {CKPT_ROOT}'
processor=AutoProcessor.from_pretrained(MODEL_ID,trust_remote_code=True,use_fast=True)
for p in checkpoints: print(' -',p)

## 3. Helpers

In [ ]:
def load_adapter(path):
    base=ModelClass.from_pretrained(MODEL_ID,trust_remote_code=True,torch_dtype=DTYPE,device_map='auto')
    return PeftModel.from_pretrained(base,str(path)).eval()
@torch.inference_mode()
def predict_one(model,image_path):
    image=Image.open(image_path).convert('RGB'); messages=[{'role':'system','content':SYSTEM_PROMPT},{'role':'user','content':[{'type':'image'},{'type':'text','text':OCR_PROMPT}]}]
    chat=processor.apply_chat_template(messages,tokenize=False,add_generation_prompt=True); inp=processor(text=[chat],images=[image],padding=True,return_tensors='pt'); device=next(model.parameters()).device
    moved={k:(v.to(device,dtype=DTYPE) if torch.is_tensor(v) and torch.is_floating_point(v) else v.to(device) if torch.is_tensor(v) else v) for k,v in inp.items()}
    torch.cuda.synchronize(); t0=time.perf_counter(); out=model.generate(**moved,use_cache=True,max_new_tokens=MAX_NEW_TOKENS,do_sample=False); torch.cuda.synchronize(); lat=time.perf_counter()-t0
    n=moved['input_ids'].shape[-1]; ids=out[0][n:].detach().cpu().tolist(); pred=processor.batch_decode([ids],skip_special_tokens=True,clean_up_tokenization_spaces=False)[0].strip(); return pred,lat

def eval_frame(model,frame,split_name):
    rows=[]; torch.cuda.reset_peak_memory_stats(); started=time.perf_counter()
    for i,(_,row) in enumerate(frame.iterrows(),1):
        pred,lat=predict_one(model,resolve_image_path(row)); gt=normalize_for_eval(row['text']); pred=normalize_for_eval(pred); rows.append({'writer_id':int(row['writer_id']),'filename':row['filename'],'ground_truth':gt,'prediction':pred,'sample_CER':float(cer(gt,pred)),'latency_sec':float(lat)})
        if i%50==0 or i==len(frame): print(f'[{i}/{len(frame)}]')
    out=pd.DataFrame(rows); m=compute_metrics(out.ground_truth.tolist(),out.prediction.tolist()); m.update({'split':split_name,'latency_mean_sec':float(out.latency_sec.mean()),'total_runtime_sec':float(time.perf_counter()-started),'peak_gpu_vram_gb':float(torch.cuda.max_memory_allocated()/1024**3)}); return out,m

## 4. Smoke each checkpoint

In [ ]:
for ckpt in checkpoints:
    print('\nSMOKE',ckpt.name); model=load_adapter(ckpt); _,m=eval_frame(model,smoke_df,'validation_smoke20'); print(m); del model; gc.collect(); torch.cuda.empty_cache()

## 5. Full validation checkpoint selection

In [ ]:
RUN_CHECKPOINT_SELECTION=True; summary=[]
if RUN_CHECKPOINT_SELECTION:
    for ckpt in checkpoints:
        model=load_adapter(ckpt); preds,m=eval_frame(model,val_df.reset_index(drop=True),'validation'); m['checkpoint']=str(ckpt); summary.append(m); preds.to_csv(RESULT_DIR/f'{ckpt.name}_val_predictions.csv',index=False); (RESULT_DIR/f'{ckpt.name}_val_metrics.json').write_text(json.dumps(m,ensure_ascii=False,indent=2),encoding='utf-8'); del model; gc.collect(); torch.cuda.empty_cache()
    summary_df=pd.DataFrame(summary).sort_values('CER').reset_index(drop=True); display(summary_df); BEST_CHECKPOINT=Path(summary_df.iloc[0].checkpoint); (RESULT_DIR/'best_checkpoint.json').write_text(json.dumps({'best_checkpoint':str(BEST_CHECKPOINT),'selection_metric':'validation CER','validation_CER':float(summary_df.iloc[0].CER)},indent=2),encoding='utf-8'); print('BEST=',BEST_CHECKPOINT)

## 6. Frozen test

In [ ]:
RUN_TEST_BENCHMARK=False
if RUN_TEST_BENCHMARK:
    if 'BEST_CHECKPOINT' not in globals(): BEST_CHECKPOINT=Path(json.loads((RESULT_DIR/'best_checkpoint.json').read_text())['best_checkpoint'])
    model=load_adapter(BEST_CHECKPOINT); preds,m=eval_frame(model,test_df.reset_index(drop=True),'test'); m['checkpoint']=str(BEST_CHECKPOINT); preds.to_csv(RESULT_DIR/'best_test_predictions.csv',index=False); (RESULT_DIR/'best_test_metrics.json').write_text(json.dumps(m,ensure_ascii=False,indent=2),encoding='utf-8'); print(json.dumps(m,ensure_ascii=False,indent=2))
else: print('Frozen test disabled.')